In [ ]:
!pip install groq

In [ ]:
import json
import time
import os
from groq import Groq

# ========== 1. CONFIGURATION ==========
# Put your Groq API key here or ensure it's in your environment variables
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY", "your_api_key_here"))

# Point this to your local JSONL training dataset
INPUT_JSONL_PATH = "/kaggle/input/datasets/teslaincarnate/training-dataset/scd_alpaca_clean2.jsonl" 
OUTPUT_JSONL_PATH = "/kaggle/working/enriched_train_dataset.jsonl"


# Using a fast, cheap model to do the heavy lifting
ENRICHMENT_MODEL = "llama-3.1-8b-instant" 

# ========== 2. ENRICHMENT PROMPT ==========
SYSTEM_PROMPT = """You are an expert medical data curator specializing in Sickle Cell Disease. 
Your job is to take a short, concise medical answer and expand it into a detailed, textbook-quality clinical explanation.
Do NOT change the underlying medical facts. Simply expand on the "why" and provide clinical context.
If the question looks like multiple choice, state the correct answer clearly and then thoroughly explain WHY it is correct.

Respond ONLY with the expanded answer text. Do not include any conversational filler."""

def enrich_answer(question, short_answer, attempt=1):
    prompt = f"QUESTION: {question}\nSHORT ANSWER: {short_answer}\n\nEXPANDED CLINICAL ANSWER:"
    
    try:
        response = groq_client.chat.completions.create(
            model=ENRICHMENT_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2, # Low temperature to prevent hallucinations
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        if "429" in str(e) or "rate limit" in str(e).lower():
            wait_time = (2 ** attempt) * 2
            print(f"  Rate limited. Waiting {wait_time} seconds...")
            time.sleep(wait_time)
            if attempt <= 5:
                return enrich_answer(question, short_answer, attempt + 1)
        print(f"  Error enriching row: {e}")
        return short_answer # Fallback to original answer if it fails

# ========== 3. RUN ENRICHMENT LOOP ==========
def process_jsonl():
    print(f"Loading dataset from {INPUT_JSONL_PATH}...")
    
    # First pass: count total lines in input so we have a progress tracker
    with open(INPUT_JSONL_PATH, 'r', encoding='utf-8') as f:
        total_rows = sum(1 for line in f)
        
    print(f"Found {total_rows} rows to enrich in total.")
    
    # --- RESUMPTION LOGIC ---
    start_index = 0
    if os.path.exists(OUTPUT_JSONL_PATH):
        # Count how many lines we have already processed
        with open(OUTPUT_JSONL_PATH, 'r', encoding='utf-8') as out_f:
            start_index = sum(1 for _ in out_f)
        print(f"Found existing progress! Resuming from row {start_index + 1}...\n")
    else:
        print("Starting fresh...\n")
        # Only clear/create the file if we are starting from scratch
        with open(OUTPUT_JSONL_PATH, 'w', encoding='utf-8') as out_f:
            pass 
            
    with open(INPUT_JSONL_PATH, 'r', encoding='utf-8') as in_f:
        for i, line in enumerate(in_f):
            # Skip rows we have already processed and saved
            if i < start_index:
                continue
                
            if not line.strip():
                continue
                
            row = json.loads(line)
            instruction = row.get("instruction", "")
            question_text = row.get("input", "")
            short_answer = row.get("output", "")
            
            print(f"Enriching [{i+1}/{total_rows}]...")
            
            # Skip enrichment if the answer is already sufficiently long (e.g., > 150 chars)
            if len(short_answer) < 150: 
                enriched_text = enrich_answer(question_text, short_answer)
                time.sleep(0.6) # Slight delay to respect API rate limits
            else:
                enriched_text = short_answer
                
            enriched_row = {
                "instruction": instruction,
                "input": question_text,
                "output": enriched_text
            }
            
            # Append the enriched row immediately (built-in checkpointing)
            with open(OUTPUT_JSONL_PATH, 'a', encoding='utf-8') as out_f:
                out_f.write(json.dumps(enriched_row) + '\n')

    print(f"\n✅ Done! Enriched dataset saved to {OUTPUT_JSONL_PATH}")

if __name__ == "__main__":
    process_jsonl()

Loading dataset from /kaggle/input/datasets/teslaincarnate/training-dataset/scd_alpaca_clean2.jsonl...
Found 3910 rows to enrich in total.
Found existing progress! Resuming from row 2712...

Enriching [2712/3910]...
Enriching [2713/3910]...
Enriching [2714/3910]...
Enriching [2715/3910]...
Enriching [2716/3910]...
Enriching [2717/3910]...
Enriching [2718/3910]...
Enriching [2719/3910]...
Enriching [2720/3910]...
Enriching [2721/3910]...
Enriching [2722/3910]...
Enriching [2723/3910]...
Enriching [2724/3910]...
Enriching [2725/3910]...
Enriching [2726/3910]...
Enriching [2727/3910]...
Enriching [2728/3910]...
Enriching [2729/3910]...
Enriching [2730/3910]...
Enriching [2731/3910]...
Enriching [2732/3910]...
Enriching [2733/3910]...
Enriching [2734/3910]...
Enriching [2735/3910]...
  Rate limited. Waiting 4 seconds...
Enriching [2736/3910]...
Enriching [2737/3910]...
Enriching [2738/3910]...
Enriching [2739/3910]...
Enriching [2740/3910]...
Enriching [2741/3910]...
Enriching [2742/3910].